In [ ]:
import os 
os.chdir(r'Q:\sachuriga\Sachuriga_Python/quattrocolo-nwb4fp\src')

from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import matplotlib.pyplot as plt
import numpy as np
import math
import pynapple as nap
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import normalize

import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap,plot_path
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,get_filed_num,unit_location_ch
from scipy.ndimage import gaussian_filter
import ast
import pandas as pd
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=np.inf)
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd

In [ ]:
import pandas as pd
df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/good_units_with_tsnLabels.pkl')
len(df_loaded)
df_good = df_loaded[df_loaded['unit_quality']=="good"]
df_py = df_good[df_good['cell_type']=="pyramidal"]

In [ ]:
import numpy as np
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

# Data
con_fis = [145, 168, 211]
exp_fis = [27, 41, 46]

con_blade = [280, 304, 318]
exp_blade = [26, 42, 91]

con_fis_and_bla = [182, 200, 237]
exp_fis_and_bla = [27, 41, 56]

# Perform ranksum tests
fis_stat, fis_p = stats.ranksums(con_fis, exp_fis,alternative='two-sided')
blade_stat, blade_p = stats.ranksums(con_blade, exp_blade,alternative='two-sided')
fis_bla_stat, fis_bla_p = stats.ranksums(con_fis_and_bla, exp_fis_and_bla,alternative='two-sided')

# Print results
print("Fis: statistic =", fis_stat, "p-value =", fis_p)
print("Blade: statistic =", blade_stat, "p-value =", blade_p)
print("Fis_and_bla: statistic =", fis_bla_stat, "p-value =", fis_bla_p)

# Prepare data for plotting
data_fis = np.concatenate([con_fis, exp_fis])
data_blade = np.concatenate([con_blade, exp_blade])
data_fis_bla = np.concatenate([con_fis_and_bla, exp_fis_and_bla])

group_fis = ['Control']*len(con_fis) + ['Experimental']*len(exp_fis)
group_blade = ['Control']*len(con_blade) + ['Experimental']*len(exp_blade)
group_fis_bla = ['Control']*len(con_fis_and_bla) + ['Experimental']*len(exp_fis_and_bla)

# Create figure with 3 subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

# Plot Fis
sns.barplot(x=group_fis, y=data_fis, ax=ax1, palette=['black', 'grey'], alpha=0.6)
sns.stripplot(x=group_fis, y=data_fis, ax=ax1, color='black', size=8)
if fis_p < 0.05:  
    ax1.set_title(f'Fis (p < 0.05)')
else:
    ax1.set_title(f'Fis (p={fis_p:.3f})')
ax1.set_ylabel('Values')
ax1.set_ylabel('Values')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['bottom'].set_visible(True)
ax1.spines['left'].set_visible(True)

# Plot Blade
sns.barplot(x=group_blade, y=data_blade, ax=ax2, palette=['black', 'grey'], alpha=0.6)
sns.stripplot(x=group_blade, y=data_blade, ax=ax2, color='black', size=8)
if blade_p< 0.05:
    ax2.set_title(f'Fis (p < 0.05)')
else:
    ax2.set_title(f'Blade (p={blade_p:.3f})')
ax2.set_ylabel('Values')
ax2.set_ylabel('Values')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['bottom'].set_visible(True)
ax2.spines['left'].set_visible(True)

# Plot Fis_and_bla
sns.barplot(x=group_fis_bla, y=data_fis_bla, ax=ax3, palette=['black', 'grey'], alpha=0.6)
sns.stripplot(x=group_fis_bla, y=data_fis_bla, ax=ax3, color='black', size=8)
if blade_p< 0.05:
    ax3.set_title(f'Fis (p < 0.05)')
else:
    ax3.set_title(f'Fis_and_bla (p={fis_bla_p:.3f})')
ax3.set_ylabel('Values')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.spines['bottom'].set_visible(True)
ax3.spines['left'].set_visible(True)

plt.tight_layout()
plt.show()

In [ ]:
df=df_py
session = "A"
if session == "Total":
    df_a = df
else:
    df_a = df[df['session'] == session]

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np

# Set global font size to 10.5
plt.rcParams.update({'font.size': 10.5})

# Load Good units
df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')
df_good = df_loaded[df_loaded['unit_quality'] == "good"]
df_py = df_good[df_good['buzaki_py_cell_type'] == "pyramidal"]

funct = [""]
for functional_plot in funct:
    if functional_plot:
        df = df_py[df_py['functional_cell_type'] == "Place cell"]
    else:
        df = df_py

    base_folder = r"Q:/sachuriga/CR_CA1_paper/Results/cell_type"

    control_ids = ['65165', '65091', '63383', '66539', '65622']
    exp_ids = ['65588', '63385', '66538', '66537', '66922']

    session = ["A", "B", "C", "Total"]

    for session in session:
        if session == "Total":
            df_a = df
        else:
            df_a = df[df['session'] == session]

        control_df = df_a[df_a['animal_id'].isin(control_ids)]
        exp_df = df_a[df_a['animal_id'].isin(exp_ids)]

        sns.set_theme(style="ticks")

        metrics = ['amplitude_median','half_width','peak_to_valley','peak_trough_ratio', 'recovery_slope', 'repolarization_slope', 
                  'mean_firing_rate',  'mean_inter_spike_interval',"mode_inter_spike_interval",
                  'bursting_index', ]

        # Define x-axis limits: (lower, upper), None for full auto, or (fixed, None)/(None, fixed) for one-sided
        xlim_dict = {
            'half_width': (None, 0.0003),              # Upper fixed, lower auto
            'recovery_slope': (None, None),     # Lower fixed, upper auto
            'repolarization_slope': None,     # Full auto
            'mean_firing_rate': (0, 8),       # Upper fixed, lower auto
            'peak_trough_ratio': (None, None), # Full auto
            'peak_to_valley': None,           # Full auto
            'bursting_index': (-1, 1),         # Upper fixed, lower auto
            'mean_inter_spike_interval': (0.1, None), # Lower fixed, upper auto
            'amplitude_median': None,          # Full auto
            'mode_inter_spike_interval': (0,0.02)
        }

        fig_violin, axes_violin = plt.subplots(2, 5, figsize=(14,7))
        axes_violin = axes_violin.flatten()

        fig_cdf, axes_cdf = plt.subplots(2, 5, figsize=(14,7))
        axes_cdf = axes_cdf.flatten()

        control_color = sns.color_palette(palette='flag')[-1]
        exp_color = "cyan"

        for idx, metric in enumerate(metrics):
            control_values = control_df[metric].dropna()
            exp_values = exp_df[metric].dropna()

            if len(control_values) > 0 and len(exp_values) > 0:
                control_mean = control_values.mean()
                exp_mean = exp_values.mean()
                control_sem = control_values.sem()
                exp_sem = exp_values.sem()

                print(f"\nComparison for {metric} (Session {session}):")
                print(f"Control mean: {control_mean:.2f} ± {control_sem:.2f}")
                print(f"Experimental mean: {exp_mean:.2f} ± {exp_sem:.2f}")

                # Original statistical tests for violin plot
                control_ks_stat, control_ks_p = stats.kstest(control_values, 'norm', 
                                                             args=(control_mean, control_values.std()))
                exp_ks_stat, exp_ks_p = stats.kstest(exp_values, 'norm', 
                                                     args=(exp_mean, exp_values.std()))
                normal = control_ks_p > 0.05 and exp_ks_p > 0.05

                levene_stat, levene_p = stats.levene(control_values, exp_values)
                homoscedastic = levene_p > 0.05

                if normal and homoscedastic:
                    t_stat, p_val = stats.ttest_ind(control_values, exp_values, equal_var=True)
                    test_name = "t-test"
                    print(f"T-test statistic: {t_stat:.2f}, p-value: {p_val:.4f}")
                else:
                    u_stat, p_val = stats.mannwhitneyu(control_values, exp_values, alternative='two-sided')
                    test_name = "Mann-Whitney U"
                    print(f"Mann-Whitney U statistic: {u_stat:.2f}, p-value: {p_val:.4f}")

                # Two-sample KS test for CDF
                ks_stat, ks_p = stats.ks_2samp(control_values, exp_values)
                print(f"KS statistic: {ks_stat:.2f}, p-value: {ks_p:.4f}")

                # Violin Plot
                plot_df = pd.DataFrame({
                    'value': pd.concat([control_values, exp_values]),
                    'group': ['Control'] * len(control_values) + ['Experimental'] * len(exp_values)
                })
                all_values = plot_df['value']
                mean_val = all_values.mean()
                std_val = all_values.std()
                plot_df_filtered = plot_df[(plot_df['value'] >= mean_val - 3 * std_val) & 
                                           (plot_df['value'] <= mean_val + 3 * std_val)]

                sns.violinplot(
                    data=plot_df_filtered, x='group', y='value', ax=axes_violin[idx],
                    palette={"Control": control_color, "Experimental": exp_color}, width=0.8, cut=0
                )
                for patch in axes_violin[idx].collections:
                    patch.set_alpha(1)

                sns.stripplot(
                    data=plot_df_filtered, x='group', y='value', ax=axes_violin[idx], size=1,
                    hue='group', palette={"Control": "black", "Experimental": "black"},
                    alpha=0.4, jitter=0.3, legend=False
                )

                #axes_violin[idx].set_title(f'{metric} Comparison')
                axes_violin[idx].set_ylabel(metric)
                axes_violin[idx].set_xlabel('')
                axes_violin[idx].yaxis.grid(False)
                axes_violin[idx].text(0.5, 0.04, f'KS p_ctrl = {control_ks_p:.4f}, p_exp = {exp_ks_p:.4f}', 
                                     ha='center', va='top', transform=axes_violin[idx].transAxes, fontsize=10.5)
                axes_violin[idx].text(0.5, 1.1, f'{test_name} p = {p_val:.4f}', 
                                     ha='center', va='top', transform=axes_violin[idx].transAxes, fontsize=10.5)
                axes_violin[idx].spines['top'].set_visible(False)
                axes_violin[idx].spines['right'].set_visible(False)
                axes_violin[idx].spines['bottom'].set_visible(True)
                axes_violin[idx].spines['left'].set_visible(True)

                # CDF Plot
                control_cfreq = stats.cumfreq(control_values, numbins=100)
                exp_cfreq = stats.cumfreq(exp_values, numbins=100)

                control_cdf = control_cfreq.cumcount / control_cfreq.cumcount[-1]
                exp_cdf = exp_cfreq.cumcount / exp_cfreq.cumcount[-1]

                num_bins = len(control_cfreq.cumcount)
                control_x = control_cfreq.lowerlimit + np.linspace(0, control_cfreq.binsize * num_bins, num_bins)
                num_bins = len(exp_cfreq.cumcount)
                exp_x = exp_cfreq.lowerlimit + np.linspace(0, exp_cfreq.binsize * num_bins, num_bins)

                axes_cdf[idx].step(control_x, control_cdf, label='Control', color=control_color, linewidth=1.5)
                axes_cdf[idx].step(exp_x, exp_cdf, label='Experimental', color=exp_color, linewidth=1.5)

                # Set x-axis limits based on xlim_dict
                xlim_setting = xlim_dict[metric]
                if xlim_setting is None or xlim_setting == (None, None):
                    min_val = min(control_values.min(), exp_values.min())
                    max_val = max(control_values.max(), exp_values.max())
                    padding = (max_val - min_val) * 0.1
                    xlim = (min_val - padding, max_val + padding)
                elif isinstance(xlim_setting, tuple):
                    min_val = min(control_values.min(), exp_values.min())
                    max_val = max(control_values.max(), exp_values.max())
                    padding = (max_val - min_val) * 0.1
                    lower = xlim_setting[0] if xlim_setting[0] is not None else min_val - padding
                    upper = xlim_setting[1] if xlim_setting[1] is not None else max_val + padding
                    xlim = (lower, upper)
                else:
                    raise ValueError(f"Invalid xlim setting for {metric}: {xlim_setting}")

                axes_cdf[idx].set_xlim(xlim)

                #axes_cdf[idx].set_title(f'{metric} CDF')
                axes_cdf[idx].set_xlabel(metric)
                axes_cdf[idx].set_ylabel('Cumulative Probability')
                axes_cdf[idx].legend(loc='lower right')
                axes_cdf[idx].grid(True, linestyle='--', alpha=0.7)
                axes_cdf[idx].text(0.5, 1.05, f'KS p = {ks_p:.4f}', 
                                  ha='center', va='bottom', transform=axes_cdf[idx].transAxes, fontsize=10.5)
                axes_cdf[idx].spines['top'].set_visible(False)
                axes_cdf[idx].spines['right'].set_visible(False)

        fig_violin.savefig(fr'{base_folder}/{session}_waveforms_pyramidal_{functional_plot}.eps', format='eps', bbox_inches='tight')
        fig_violin.savefig(fr'{base_folder}/{session}_waveforms_pyramidal_{functional_plot}.png', format='png', bbox_inches='tight')

        fig_cdf.savefig(fr'{base_folder}/{session}_waveforms_pyramidal_{functional_plot}_cdf.eps', format='eps', bbox_inches='tight')
        fig_cdf.savefig(fr'{base_folder}/{session}_waveforms_pyramidal_{functional_plot}_cdf.png', format='png', bbox_inches='tight')

        plt.tight_layout()
        plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
##Load Good units

df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')
len(df_loaded)
df_good = df_loaded[df_loaded['unit_quality']=="good"]
df_py = df_good[df_good['cell_type']=="interneuron"]
df=df_py


base_folder = r"Q:/sachuriga/CR_CA1_paper/Results/cell_type"

control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

session = ["A","B","C","Total"]

for session in session:
    # Filter for 'A' sessions
    if session == "Total":
        df_a = df
    else:
        df_a = df[df['session'] == session]

    # Separate into control and experimental groups
    control_df = df_a[df_a['animal_id'].isin(control_ids)]
    exp_df = df_a[df_a['animal_id'].isin(exp_ids)]

    # Set Seaborn theme
    sns.set_theme(style="ticks")

    # Statistical comparisons for scalar metrics
    metrics = ['half_width', 'recovery_slope', 'repolarization_slope', 
            'firing_range', 'peak_trough_ratio', 'peak_to_valley', 'bursting_index',"mean_inter_spike_interval", 'amplitude_median']

    # Create figure with 3x2 subplots
    fig, axes = plt.subplots(3, 3, figsize=(14, 14))
    axes = axes.flatten()  # Flatten the 2D array of axes for easier iteration

    # Define custom colors
    control_color = "red"  # Dark blue for Control
    #exp_color = sns.color_palette(palette='gist_rainbow')[3]       # Light blue for Experimental
    exp_color = "magenta"
    for idx, metric in enumerate(metrics):
        control_values = control_df[metric].dropna()
        exp_values = exp_df[metric].dropna()
        
        if len(control_values) > 0 and len(exp_values) > 0:
            control_mean = control_values.mean()
            exp_mean = exp_values.mean()
            control_sem = control_values.sem()
            exp_sem = exp_values.sem()
            
            print(f"\nComparison for {metric}:")
            print(f"Control mean: {control_mean:.2f} ± {control_sem:.2f}")
            print(f"Experimental mean: {exp_mean:.2f} ± {exp_sem:.2f}")
            
            control_ks_stat, control_ks_p = stats.kstest(control_values, 'norm', 
                                                            args=(control_mean, control_values.std()))
            exp_ks_stat, exp_ks_p = stats.kstest(exp_values, 'norm', 
                                                    args=(exp_mean, exp_values.std()))
            normal = control_ks_p > 0.05 and exp_ks_p > 0.05

            # Homoscedasticity test (Levene’s test)
            levene_stat, levene_p = stats.levene(control_values, exp_values)
            homoscedastic = levene_p > 0.05

            # Choose statistical test based on normality and homoscedasticity
            if normal and homoscedastic:
                # Use t-test
                t_stat, p_val = stats.ttest_ind(control_values, exp_values, equal_var=True)
                test_name = "t-test"
                print(f"T-test statistic: {t_stat:.2f}, p-value: {p_val:.4f}")
            else:
                # Use Mann-Whitney U test
                u_stat, p_val = stats.mannwhitneyu(control_values, exp_values, alternative='two-sided')
                test_name = "Mann-Whitney U"
                print(f"Mann-Whitney U statistic: {u_stat:.2f}, p-value: {p_val:.4f}")

            
            # Prepare data for Seaborn plotting
            plot_df = pd.DataFrame({
                'value': pd.concat([control_values, exp_values]),
                'group': ['Control'] * len(control_values) + ['Experimental'] * len(exp_values)
            })
            
            # Check if deviation is "too large" (using coefficient of variation > 1 as threshold)
            all_values = plot_df['value']
            cv = all_values.std() / all_values.mean()  # Coefficient of variation
            use_log_scale = cv > 1 and all_values.min() > 0  # Ensure positive values for log scale
            
            # Filter out outliers (e.g., beyond 3 standard deviations)
            mean_val = all_values.mean()
            std_val = all_values.std()
            plot_df_filtered = plot_df[(plot_df['value'] >= mean_val - 3 * std_val) & 
                                    (plot_df['value'] <= mean_val + 3 * std_val)]
            
            # Create violin plot on the specific subplot
            violin = sns.violinplot(
                data=plot_df_filtered,
                x='group',
                y='value',
                ax=axes[idx],
                palette={"Control": control_color, "Experimental": exp_color},
                width=0.8,
                cut=0  # Prevents violin tails from extending beyond data range
            )
            
            # Set alpha (transparency) for the violin plot
            for patch in violin.collections:
                patch.set_alpha(1)
            
            # Add individual points with matching colors
            sns.stripplot(
                data=plot_df_filtered,
                x='group',
                y='value',
                ax=axes[idx],
                size=4,
                hue='group',
                palette={"Control": "black", "Experimental": "black"},
                alpha=0.4,
                jitter=0.3,
                legend=False
            )
            
            # Set title and labels
            axes[idx].set_title(f'{metric} Comparison')
            axes[idx].set_ylabel(metric)
            axes[idx].set_xlabel('Group')
            axes[idx].yaxis.grid(False)
            axes[idx].set(xlabel="")
            
            # Apply log scale if deviation is too large
            if use_log_scale:
                axes[idx].set_yscale('log')
                axes[idx].set_ylabel(f'{metric} (log scale)')

            axes[idx].text(0.5, 0.04, f'KS p_ctrl = {control_ks_p:.4f}, p_exp = {exp_ks_p:.4f}', 
                horizontalalignment='center', verticalalignment='top', 
                transform=axes[idx].transAxes, fontsize=8)
            
            # Add p-value at the top of the plot
            axes[idx].text(0.5, 1.1, f'{test_name} p = {p_val:.4f}', 
                            horizontalalignment='center', verticalalignment='top', 
                            transform=axes[idx].transAxes, fontsize=10)
            # # Add p-value at the top of the plot
            # axes[idx].text(0.5, 0.95, f'p = {p_val:.4f}', 
            #             horizontalalignment='center', 
            #             verticalalignment='top', 
            #             transform=axes[idx].transAxes, 
            #             fontsize=10)
            
            # Remove top and right spines, keep bottom (x) and left (y) axes
            axes[idx].spines['top'].set_visible(False)
            axes[idx].spines['right'].set_visible(False)
            axes[idx].spines['bottom'].set_visible(True)
            axes[idx].spines['left'].set_visible(True)

    # Save the figure
    fig.savefig(fr'{base_folder}/{session}_waveforms_interneuron.eps', format='eps', bbox_inches='tight')
    fig.savefig(fr'{base_folder}/{session}_waveforms_interneuron.png', format='png', bbox_inches='tight')

    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
##Load Good units

df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')
len(df_loaded)
df_good = df_loaded[df_loaded['unit_quality']=="good"]
df_py = df_good[df_good['cell_type']=="wid_interneuron?"]
df=df_py


base_folder = r"Q:/sachuriga/CR_CA1_paper/Results/cell_type"

control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

session = ["A","B","C","Total"]

for session in session:
    # Filter for 'A' sessions
    if session == "Total":
        df_a = df
    else:
        df_a = df[df['session'] == session]

    # Separate into control and experimental groups
    control_df = df_a[df_a['animal_id'].isin(control_ids)]
    exp_df = df_a[df_a['animal_id'].isin(exp_ids)]

    # Set Seaborn theme
    sns.set_theme(style="ticks")

    # Statistical comparisons for scalar metrics
    metrics = ['half_width', 'recovery_slope', 'repolarization_slope', 
            'firing_range', 'peak_trough_ratio', 'peak_to_valley', 'bursting_index',"mean_inter_spike_interval"]

    # Create figure with 3x2 subplots
    fig, axes = plt.subplots(3, 3, figsize=(14, 14))
    axes = axes.flatten()  # Flatten the 2D array of axes for easier iteration

    # Define custom colors
    control_color = "green"  # Changed from red to green
    exp_color = "lightgreen"  # Changed from magenta to lightgreen
    
    for idx, metric in enumerate(metrics):
        control_values = control_df[metric].dropna()
        exp_values = exp_df[metric].dropna()
        
        if len(control_values) > 0 and len(exp_values) > 0:
            control_mean = control_values.mean()
            exp_mean = exp_values.mean()
            control_sem = control_values.sem()
            exp_sem = exp_values.sem()
            
            print(f"\nComparison for {metric}:")
            print(f"Control mean: {control_mean:.2f} ± {control_sem:.2f}")
            print(f"Experimental mean: {exp_mean:.2f} ± {exp_sem:.2f}")
            
            control_ks_stat, control_ks_p = stats.kstest(control_values, 'norm', 
                                                            args=(control_mean, control_values.std()))
            exp_ks_stat, exp_ks_p = stats.kstest(exp_values, 'norm', 
                                                    args=(exp_mean, exp_values.std()))
            normal = control_ks_p > 0.05 and exp_ks_p > 0.05

            # Homoscedasticity test (Levene’s test)
            levene_stat, levene_p = stats.levene(control_values, exp_values)
            homoscedastic = levene_p > 0.05

            # Choose statistical test based on normality and homoscedasticity
            if normal and homoscedastic:
                # Use t-test
                t_stat, p_val = stats.ttest_ind(control_values, exp_values, equal_var=True)
                test_name = "t-test"
                print(f"T-test statistic: {t_stat:.2f}, p-value: {p_val:.4f}")
            else:
                # Use Mann-Whitney U test
                u_stat, p_val = stats.mannwhitneyu(control_values, exp_values, alternative='two-sided')
                test_name = "Mann-Whitney U"
                print(f"Mann-Whitney U statistic: {u_stat:.2f}, p-value: {p_val:.4f}")

            
            # Prepare data for Seaborn plotting
            plot_df = pd.DataFrame({
                'value': pd.concat([control_values, exp_values]),
                'group': ['Control'] * len(control_values) + ['Experimental'] * len(exp_values)
            })
            
            # Check if deviation is "too large" (using coefficient of variation > 1 as threshold)
            all_values = plot_df['value']
            cv = all_values.std() / all_values.mean()  # Coefficient of variation
            use_log_scale = cv > 1 and all_values.min() > 0  # Ensure positive values for log scale
            
            # Filter out outliers (e.g., beyond 3 standard deviations)
            mean_val = all_values.mean()
            std_val = all_values.std()
            plot_df_filtered = plot_df[(plot_df['value'] >= mean_val - 3 * std_val) & 
                                    (plot_df['value'] <= mean_val + 3 * std_val)]
            
            # Create violin plot on the specific subplot
            violin = sns.violinplot(
                data=plot_df_filtered,
                x='group',
                y='value',
                ax=axes[idx],
                palette={"Control": control_color, "Experimental": exp_color},
                width=0.8,
                cut=0  # Prevents violin tails from extending beyond data range
            )
            
            # Set alpha (transparency) for the violin plot
            for patch in violin.collections:
                patch.set_alpha(1)
            
            # Add individual points with matching colors
            sns.stripplot(
                data=plot_df_filtered,
                x='group',
                y='value',
                ax=axes[idx],
                size=4,
                hue='group',
                palette={"Control": "black", "Experimental": "black"},
                alpha=0.4,
                jitter=0.3,
                legend=False
            )
            
            # Set title and labels
            axes[idx].set_title(f'{metric} Comparison')
            axes[idx].set_ylabel(metric)
            axes[idx].set_xlabel('Group')
            axes[idx].yaxis.grid(False)
            axes[idx].set(xlabel="")
            
            # Apply log scale if deviation is too large
            if use_log_scale:
                axes[idx].set_yscale('log')
                axes[idx].set_ylabel(f'{metric} (log scale)')

            axes[idx].text(0.5, 0.04, f'KS p_ctrl = {control_ks_p:.4f}, p_exp = {exp_ks_p:.4f}', 
                horizontalalignment='center', verticalalignment='top', 
                transform=axes[idx].transAxes, fontsize=8)
            
            # Add p-value at the top of the plot
            axes[idx].text(0.5, 1.1, f'{test_name} p = {p_val:.4f}', 
                            horizontalalignment='center', verticalalignment='top', 
                            transform=axes[idx].transAxes, fontsize=10)
            
            # Remove top and right spines, keep bottom (x) and left (y) axes
            axes[idx].spines['top'].set_visible(False)
            axes[idx].spines['right'].set_visible(False)
            axes[idx].spines['bottom'].set_visible(True)
            axes[idx].spines['left'].set_visible(True)

    # Save the figure
    fig.savefig(fr'{base_folder}/{session}_waveforms_wide_interneuron.eps', format='eps', bbox_inches='tight')
    fig.savefig(fr'{base_folder}/{session}_waveforms_wide_interneuron.png', format='png', bbox_inches='tight')

    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np

df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')
# [Previous data loading code remains the same until the session loop]
df_good = df_loaded[df_loaded['unit_quality']=="good"]
df_py = df_good[df_good['buzaki_py_cell_type']=="pyramidal"]

base_folder = r"Q:/sachuriga/CR_CA1_paper/Results/functional_cell_type"
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

def remove_outliers(df, column):
    """Remove outliers using IQR method"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

funct = ["Place cells",""]


for functional_plot in funct:
    if functional_plot :
        df = df_py[df_py['functional_cell_type']=="Place cell"]
    else:
        df = df_py
    session = ["A","B","C","Total"]
    for session in session:
        # Filter for sessions
        if session == "Total":
            df_a = df
        else:
            df_a = df[df['session'] == session]

        # Separate into control and experimental groups
        control_df = df_a[df_a['animal_id'].isin(control_ids)]
        exp_df = df_a[df_a['animal_id'].isin(exp_ids)]

        # Set Seaborn theme
        sns.set_theme(style="ticks")

        # Metrics for analysis
        metrics = ['half_width', 'recovery_slope', 'repolarization_slope', 
                'firing_range', 'peak_trough_ratio', 'peak_to_valley']
        
        metrics1 = ['matlab_test_stat_si']
        
        # Create figure with 3x2 subplots
        fig, axes = plt.subplots(2,3, figsize=(14, 9))
        axes = axes.flatten()

        # Define custom colors
        control_color = "blue"
        exp_color = "cyan"

        for idx, metric in enumerate(metrics):
            # Prepare and clean control group data
            control_data = pd.concat([control_df[metrics1[0]], control_df[metric]], axis=1).dropna()
            control_data = remove_outliers(control_data, metrics1[0])
            control_data = remove_outliers(control_data, metric)
            
            # Prepare and clean experimental group data
            exp_data = pd.concat([exp_df[metrics1[0]], exp_df[metric]], axis=1).dropna()
            exp_data = remove_outliers(exp_data, metrics1[0])
            exp_data = remove_outliers(exp_data, metric)

            if len(control_data) > 2 and len(exp_data) > 2:  # Need at least 3 points for meaningful fit
                # Linear regression for control
                control_slope, control_intercept, control_r, control_p, control_se = stats.linregress(
                    control_data[metrics1[0]], control_data[metric])
                
                # Linear regression for experimental
                exp_slope, exp_intercept, exp_r, exp_p, exp_se = stats.linregress(
                    exp_data[metrics1[0]], exp_data[metric])

                # Print results
                print(f"\n{session} - {metric} (after outlier removal):")
                print(f"Control: slope = {control_slope:.4f} ± {control_se:.4f}, R² = {control_r**2:.4f}, n = {len(control_data)}")
                print(f"Experimental: slope = {exp_slope:.4f} ± {exp_se:.4f}, R² = {exp_r**2:.4f}, n = {len(exp_data)}")

                # Test difference in slopes (using z-test)
                se_diff = np.sqrt(control_se**2 + exp_se**2)
                z_stat = (control_slope - exp_slope) / se_diff
                p_val = stats.norm.sf(abs(z_stat)) * 2  # Two-tailed test

                print(f"Slope difference z-statistic: {z_stat:.4f}, p-value: {p_val:.4f}")

                # Create scatter plot with regression lines
                # Control group
                control_plot = sns.scatterplot(
                    data=control_data, 
                    x=metrics1[0], 
                    y=metric, 
                    ax=axes[idx], 
                    color=control_color,
                    label='Control'
                )
                control_line = axes[idx].plot(control_data[metrics1[0]], 
                                            control_slope * control_data[metrics1[0]] + control_intercept, 
                                            color=control_color,
                                            label=f'Control (R² = {control_r**2:.2f})')

                # Experimental group
                exp_plot = sns.scatterplot(
                    data=exp_data, 
                    x=metrics1[0], 
                    y=metric, 
                    ax=axes[idx], 
                    color=exp_color,
                    label='Experimental'
                )
                exp_line = axes[idx].plot(exp_data[metrics1[0]], 
                                        exp_slope * exp_data[metrics1[0]] + exp_intercept, 
                                        color=exp_color,
                                        label=f'Exp (R² = {exp_r**2:.2f})')

                # Customize plot
                axes[idx].set_title(f'{metric} vs {metrics1[0]}')
                axes[idx].set_xlabel(metrics1[0])
                axes[idx].set_ylabel(metric)
                
                # Update legend with R² values
                axes[idx].legend(handles=[control_line[0], exp_line[0]],
                            labels=[f'Control (R² = {control_r**2:.2f})', 
                                    f'Exp (R² = {exp_r**2:.2f})'])
                
                # Add p-value
                axes[idx].text(0.5, 0.95, f'p = {p_val:.4f}', 
                            horizontalalignment='center', 
                            verticalalignment='top', 
                            transform=axes[idx].transAxes, 
                            fontsize=10)

                # Remove top and right spines
                axes[idx].spines['top'].set_visible(False)
                axes[idx].spines['right'].set_visible(False)

        # Save the figure
        fig.savefig(fr'{base_folder}/{session}_linear_fits_pyramidal_no_outliers_{functional_plot}.eps', format='eps', bbox_inches='tight')
        fig.savefig(fr'{base_folder}/{session}_linear_fits_pyramidal_no_outliers_{functional_plot}.png', format='png', bbox_inches='tight')

        # Adjust layout and show
        plt.tight_layout()
        plt.show()

In [ ]:
import os
os.chdir(r"Q:/sachuriga/Sachuriga_Python/quattrocolo-nwb4fp/src")

import sys
from pathlib import Path

import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import UnitMatchPy.extract_raw_data as erd
import numpy as np 
import probeinterface as pi
import spikeinterface.exporters as sex
from spikeinterface.preprocessing import (bandpass_filter,
                                           common_reference,
                                           whiten)
from spikeinterface.extractors.neoextractors.openephys import OpenEphysBinaryRecordingExtractor
import os
import glob
from nwb4fp.data.helpers import unit_location_ch
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'Unit_match')))
from unit_match_files import load_filen ,run_unitmatch


#plot waveform template
import pandas as pd
import numpy as np
df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')
uniques = np.unique(df_loaded['session_id'])


In [ ]:
base_fodler="S:\Sachuriga\Ephys_Recording\CR_CA1"
new_df=[]
for u in uniques:
    file = u.split(".nwb")[0]
    animal =u.split(".nwb")[0].split("_")[0]
    temp = df_loaded[df_loaded['session_id']==u].reset_index()
    rec_path = fr"{base_fodler}/{animal}/{file}"
    rp = [rec_path.split("_phy_k_manual")[0]]
    stream_name  = OpenEphysBinaryRecordingExtractor(rp[0],stream_id='0').get_streams(rp[0])[0][0]
    print(fr"Merging step_Before mannual search the stream_name. Auto search result is {stream_name}")
    record_node = stream_name.split("#")[0]
    aquisition_sys = stream_name.split("#")[1]
    recording= se.read_openephys(Path(rp[0]), stream_name=stream_name, load_sync_timestamps=True)

        
    manufacturer = 'cambridgeneurotech'
    probe_name = 'ASSY-236-F'
    probe = pi.get_probe(manufacturer, probe_name)
    print(probe)
    # probe.wiring_to_device('cambridgeneurotech_mini-amp-64')
    # map channels to device indices
    mapping_to_device = [
        # connector J2 TOP
        41, 39, 38, 37, 35, 34, 33, 32, 29, 30, 28, 26, 25, 24, 22, 20,
        46, 45, 44, 43, 42, 40, 36, 31, 27, 23, 21, 18, 19, 17, 16, 14,
        # connector J1 BOTTOM
        55, 53, 54, 52, 51, 50, 49, 48, 47, 15, 13, 12, 11, 9, 10, 8,
        63, 62, 61, 60, 59, 58, 57, 56, 7, 6, 5, 4, 3, 2, 1, 0
    ]
    probe.set_device_channel_indices(mapping_to_device)
    probe.to_dataframe(complete=True).loc[:, ["contact_ids", "shank_ids", "device_channel_indices"]]
    probegroup = pi.ProbeGroup()
    probegroup.add_probe(probe)
    pi.write_prb(f"{probe_name}.prb", probegroup, group_mode="by_shank")
    recording_prb = recording.set_probe(probe, group_mode="by_shank")
    sorting = se.read_phy(Path(rec_path))
    recording  = spre.bandpass_filter(recording_prb , freq_min=600, freq_max=8000) #highpass
    recording = spre.common_reference(recording=recording, operator="median", reference="global")
    recording = whiten(recording, int_scale=200, mode='local', radius_um=100.0)
    id = temp['unit_name']
    sorting = sorting.select_units(np.int32(id.values[:]))
    analyzer = si.create_sorting_analyzer(sorting, recording, sparse=False)
    analyzer.compute(
                "random_spikes","waveforms",
                method="uniform",
                max_spikes_per_unit=1000)
    analyzer.compute("noise_levels")
    analyzer.compute("templates")
    templates = analyzer.get_extension('templates')
    av_wave =  templates.get_data()
    wavs = []
    for i in range(len(temp)):

        x = temp['x'][i]
        y = temp['y'][i]
        num=unit_location_ch(x=np.float64(x),y=np.float64(y))
        wavs.append(av_wave[i,:,num])

        # print(num)
        # if temp['cell_type'][i]=="pyramidal":
        #     plt.plot(av_wave[i,:,num],color="blue", alpha=0.5)
        # elif temp['cell_type'][i]=="interneuron":
        #     plt.plot(av_wave[i,:,num],color="red")
        # else:
        #     plt.plot(av_wave[i,:,num],color="green")

    temp['waveform']=wavs
    new_df.append(temp)

all_units_df = pd.concat(new_df, ignore_index=True)
all_units_output_file = os.path.join(r"Q:\sachuriga\CR_CA1_paper\tables/", "all_units_table_with_waveforms.pkl")
all_units_df.to_pickle(all_units_output_file)

In [ ]:
all_units_df = pd.concat(new_df, ignore_index=True)
all_units_output_file = os.path.join(r"Q:\sachuriga\CR_CA1_paper\tables/", "all_units_table_with_waveforms.pkl")
all_units_df.to_pickle(all_units_output_file)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
all_units_df=pd.read_pickle(r"Q:\sachuriga\CR_CA1_paper\tables/all_units_table_with_waveforms.pkl")

control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

# Function to normalize waveforms
def normalize_waveform(wave):
    min_val = np.min(wave)
    return wave / abs(min_val)

# Filter data for control and experimental groups
control_pyramidal = all_units_df[(all_units_df['animal_id'].isin(control_ids)) & (all_units_df['buzaki_cell_type'] == 'pyramidal')]
exp_pyramidal = all_units_df[(all_units_df['animal_id'].isin(exp_ids)) & (all_units_df['buzaki_cell_type'] == 'pyramidal')]

control_interneuron = all_units_df[(all_units_df['animal_id'].isin(control_ids)) & (all_units_df['buzaki_cell_type'] == 'narrow_spike_interneurons')]
exp_interneuron = all_units_df[(all_units_df['animal_id'].isin(exp_ids)) & (all_units_df['buzaki_cell_type'] == 'narrow_spike_interneurons')]

# Normalize waveforms
control_pyramidal['normalized_waveform'] = control_pyramidal['waveform'].apply(normalize_waveform)
exp_pyramidal['normalized_waveform'] = exp_pyramidal['waveform'].apply(normalize_waveform)
control_interneuron['normalized_waveform'] = control_interneuron['waveform'].apply(normalize_waveform)
exp_interneuron['normalized_waveform'] = exp_interneuron['waveform'].apply(normalize_waveform)

# Calculate median waveforms
median_control_pyramidal = np.mean(np.stack(control_pyramidal['normalized_waveform'].tolist()), axis=0)
median_exp_pyramidal = np.mean(np.stack(exp_pyramidal['normalized_waveform'].tolist()), axis=0)

median_control_interneuron = np.mean(np.stack(control_interneuron['normalized_waveform'].tolist()), axis=0)
median_exp_interneuron = np.mean(np.stack(exp_interneuron['normalized_waveform'].tolist()), axis=0)

# Assuming 'all_units_df' is your DataFrame with 'cell_type' and 'waveform' columns
# First figure: Original waveforms
fig = plt.figure(figsize=(20, 10))  # Increased width to accommodate two subplots

# Subplot 1: Original pyramidal cells
plt.subplot(1, 2, 1)
py = all_units_df[all_units_df['buzaki_cell_type'] == "pyramidal"]
for wave in py['waveform']:
    if np.abs(np.min(wave)) > np.abs(np.max(wave)):
        plt.plot(wave, color="blue", alpha=0.1)
plt.title('Original Pyramidal Waveforms')
plt.xlabel('Time')
plt.ylabel('Amplitude')
#plt.axis('off')  # Remove axes
# Add a vertical scale bar for 100 units
plt.plot([85, 85], [-1400, -1900], 'k-', linewidth=2)  # Black line, 2 pixels wide
plt.text(87, -1700, '500', verticalalignment='center')  # Add text label
plt.axis('off')  # Remove axes

# # Subplot 2: Normalized pyramidal cells
plt.subplot(1, 2, 2)

for wave in control_pyramidal['normalized_waveform']:
    #if np.abs(np.min(wave)) > np.abs(np.max(wave)):
    plt.plot(wave, color='blue', alpha=0.01)

for wave in exp_pyramidal['normalized_waveform']:
   #if np.abs(np.min(wave)) > np.abs(np.max(wave)):
    plt.plot(wave, color='cyan', alpha=0.01)

plt.title('Normalized Pyramidal Waveforms (min=-1)')  # Corrected title
plt.xlabel('Time')
plt.ylabel('Normalized Amplitude')
# Plot median pyramidal cell waveforms
plt.plot(median_control_pyramidal, linewidth=2, color='darkblue')
plt.plot(median_exp_pyramidal, linewidth=2, color='darkcyan')
plt.tight_layout()
plt.axis('off')  # Remove axes
plt.ylim(-1.5625, 0.9375)
plt.savefig(r'Q:\sachuriga/CR_CA1_paper/Results/Cell_type/waveforms_pyramidal.png')

# Second figure: Interneurons
fig2 = plt.figure(figsize=(20, 10))

# Subplot 1: Original interneurons
plt.subplot(1, 2, 1)
py = all_units_df[all_units_df['buzaki_cell_type'] == "narrow_spike_interneurons"]
for wave in py['waveform']:
    if np.abs(np.min(wave)) > np.abs(np.max(wave)):
        plt.plot(wave, color="red", alpha=0.1)
plt.title('Original Interneuron Waveforms')
plt.xlabel('Time')
plt.ylabel('Amplitude')
#plt.axis('off')  # Remove axes
plt.plot([85, 85], [-200, -700], 'k-', linewidth=2)  # Black line, 2 pixels wide
plt.text(87, -450, '500', verticalalignment='center')  # Add text label
plt.axis('off')  # Remove axes

# Subplot 2: Normalized interneurons
plt.subplot(1, 2, 2)
# for wave in py['waveform']:
#     if np.abs(np.min(wave)) > np.abs(np.max(wave)):
#         # Normalize by the minimum value to make it -1
#         min_val = np.min(wave)
#         normalized_wave = wave / abs(min_val)  # Divide by the absolute value to make the minimum -1
#         plt.plot(normalized_wave, color="red", alpha=0.1)
# Plot individual interneuron traces
for wave in control_interneuron['normalized_waveform']:
    plt.plot(wave, color='red', alpha=0.05)

for wave in exp_interneuron['normalized_waveform']:
    plt.plot(wave, color='magenta', alpha=0.05)
# Plot median interneuron waveforms
plt.plot(median_control_interneuron, linewidth=2, color='darkred')
plt.plot(median_exp_interneuron, linewidth=2, color='darkmagenta')
plt.ylim(-1.5625, 0.65)
plt.title('Normalized Interneuron Waveforms (min=-1)')  # Corrected title
plt.xlabel('Time')
plt.ylabel('Normalized Amplitude')
plt.axis('off')  # Remove axes
plt.tight_layout()
plt.savefig(r'Q:\sachuriga/CR_CA1_paper/Results/Cell_type/waveforms_interneurons.png')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Assuming all_units_df is already defined
# all_units_df = ...  # Replace with your DataFrame loading if needed

control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

# Function to normalize waveforms
def normalize_waveform(wave):
    min_val = np.min(wave)
    return wave / abs(min_val)

# Filter data for control and experimental groups
control_pyramidal = all_units_df[(all_units_df['animal_id'].isin(control_ids)) & (all_units_df['buzaki_cell_type'] == 'pyramidal')]
exp_pyramidal = all_units_df[(all_units_df['animal_id'].isin(exp_ids)) & (all_units_df['buzaki_cell_type'] == 'pyramidal')]

control_interneuron = all_units_df[(all_units_df['animal_id'].isin(control_ids)) & (all_units_df['buzaki_cell_type'] == 'narrow_spike_interneurons')]
exp_interneuron = all_units_df[(all_units_df['animal_id'].isin(exp_ids)) & (all_units_df['buzaki_cell_type'] == 'narrow_spike_interneurons')]

# Normalize waveforms
control_pyramidal['normalized_waveform'] = control_pyramidal['waveform'].apply(normalize_waveform)
exp_pyramidal['normalized_waveform'] = exp_pyramidal['waveform'].apply(normalize_waveform)
control_interneuron['normalized_waveform'] = control_interneuron['waveform'].apply(normalize_waveform)
exp_interneuron['normalized_waveform'] = exp_interneuron['waveform'].apply(normalize_waveform)

# Calculate median waveforms
median_control_pyramidal = np.mean(np.stack(control_pyramidal['normalized_waveform'].tolist()), axis=0)
median_exp_pyramidal = np.mean(np.stack(exp_pyramidal['normalized_waveform'].tolist()), axis=0)

median_control_interneuron = np.mean(np.stack(control_interneuron['normalized_waveform'].tolist()), axis=0)
median_exp_interneuron = np.mean(np.stack(exp_interneuron['normalized_waveform'].tolist()), axis=0)

# Plotting individual traces and median
plt.figure(figsize=(10, 6))

# Plot individual pyramidal cell traces
for wave in control_pyramidal['normalized_waveform']:
    plt.plot(wave, color='blue', alpha=0.05)

for wave in exp_pyramidal['normalized_waveform']:
    plt.plot(wave, color='cyan', alpha=0.05)

# Plot median pyramidal cell waveforms
plt.plot(median_control_pyramidal, linewidth=2, color='darkblue')
plt.plot(median_exp_pyramidal, linewidth=2, color='darkcyan')
plt.ylim(-1.5625, 0.9375)
plt.title('Normalized Pyramidal Waveforms with Median')
plt.xlabel('Time')
plt.ylabel('Normalized Amplitude')
plt.legend()
plt.tight_layout()
plt.show()



plt.figure(figsize=(10, 6))

# Plot individual interneuron traces
for wave in control_interneuron['normalized_waveform']:
    plt.plot(wave, color='red', alpha=0.05)

for wave in exp_interneuron['normalized_waveform']:
    plt.plot(wave, color='magenta', alpha=0.05)

# Plot median interneuron waveforms
plt.plot(median_control_interneuron, linewidth=2, color='darkred')
plt.plot(median_exp_interneuron, linewidth=2, color='darkmagenta')

plt.title('Normalized Interneuron Waveforms with Median')
plt.xlabel('Time')
plt.ylabel('Normalized Amplitude')
plt.tight_layout()
plt.show()